### Interactive Growth Map

A growth map of a Mapper graph is a visualization that displays each topics relative size and growth. It is inspired by a [similar visualization often used to display stock market data](https://stockcharts.com/marketcarpet/).

Let's demonstrate a growth map by fitting a `TemporalMapper` to a small dataset of 10,000 arXiv machine learning papers. The paper's titles and abstracts were concatenated and embedded using the sentence transformer [all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2), and then reduced to 2D with UMAP.

In [1]:
import temporalmapper as tm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, io
from sklearn.cluster import DBSCAN
from fast_hdbscan import HDBSCAN
import datamapplot as dmp
response = requests.get(
    'https://github.com/TutteInstitute/temporal-mapper/raw/refs/heads/docs/docs/data/ai_arxiv_coordinates.npy'
)
map_data = np.load(io.BytesIO(response.content))

response = requests.get(
    'https://github.com/TutteInstitute/temporal-mapper/raw/refs/heads/docs/docs/data/ai_arxiv_data.feather'
)
df = pd.read_feather(io.BytesIO(response.content))

df.head()

,title,abstract,id,created,authors,arxiv,doi
0,automated rating of recorded classroom present...,effective presentation skills can help to succ...,1801.00453,2018-01-01,"[akzharkyn izbassarova, aidana irmanova, a. p....",cs.ai,10.1109/icacci.2017.8125872
1,accelerating deep learning with memcomputing,restricted boltzmann machines (rbms) and their...,1801.00512,2018-01-01,"[haik manukian, fabio l. traversa, massimilian...",cs.ai,
2,accelerating deep learning with memcomputing,restricted boltzmann machines (rbms) and their...,1801.00512,2018-01-01,"[haik manukian, fabio l. traversa, massimilian...",cs.lg,
3,accurate reconstruction of image stimuli from ...,"in neuroscience, all kinds of computation mode...",1801.00602,2018-01-02,"[kai qiao, chi zhang, linyuan wang, bin yan, j...",cs.ai,
4,deep learning: a critical appraisal,although deep learning has historical roots go...,1801.00631,2018-01-02,[gary marcus],cs.lg,


In [2]:
# Compute a time column T which is the number of days since Jan 01, 2018.
def date_to_T(date):
    d0 = pd.Timestamp('2018-01-01')
    delta = date-d0
    return delta.days

df["date"] = pd.to_datetime(df["created"])
df["T"] = df["date"].apply(
    lambda x: date_to_T(x)
)
time = df["T"].to_numpy()

clusterer = HDBSCAN(
    cluster_selection_method='eom',
    min_cluster_size=20,
)
mapper = tm.TemporalMapper(
    time,
    map_data,
    clusterer,
    slice_method = 'data',
    N_checkpoints = 8,
    kernel=tm.kernels.square
)
mapper.fit()

,time,"array([ 0, ...hape=(10000,))"
,data,array([[-0.19...dtype=float32)
,clusterer,HDBSCAN(min_cluster_size=20)
,N_checkpoints,8
,neighbours,50
,overlap,0.5
,inclusion_threshold,0.01
,checkpoints,"array([ 66, 1...01, 372, 428])"
,show_outliers,False
,slice_method,'data'
,rate_sensitivity,1


Now that we've fit a Mapper graph, we can use `tm.plotting.growth_map` to generate a growth map. The size of each square indicates the number of data points in the corresponding topic, and it's colour represents the growth of that topic.

In [ ]:
tm.plotting.growth_map(mapper)


By default, growth_map displays topics across the entire time range, but we can pass an index parameter to show only the topics at a certain time slice.

In [4]:
tm.plotting.growth_map(mapper, index=2)